In [62]:
# ============================================================
# Game Simulator
# Uses trained pregame PA model to simulate full MLB games
# ============================================================

import numpy as np
import pandas as pd

from catboost import CatBoostClassifier

In [63]:
# ----------------------------
# Load trained pregame PA model
# ----------------------------

model = CatBoostClassifier()
model.load_model("models/pregame_pa_model.cbm")

print("Pregame PA model loaded successfully")

Pregame PA model loaded successfully


In [64]:
# ----------------------------
# Model feature order
# ----------------------------

features = [
    "pitcher",
    "batter",
    "p_throws",
    "stand",
    "inning",
    "outs_when_up",
    "balls",
    "strikes",
    "on_1b",
    "on_2b",
    "on_3b"
]

print(features)

['pitcher', 'batter', 'p_throws', 'stand', 'inning', 'outs_when_up', 'balls', 'strikes', 'on_1b', 'on_2b', 'on_3b']


In [65]:
# ----------------------------
# Outcome classes
# ----------------------------

classes = model.classes_
print("Model classes:", classes)

Model classes: ['double' 'hr' 'out' 'single' 'strikeout' 'triple' 'walk']


In [66]:
# ----------------------------
# Predict one plate appearance
# ----------------------------

def predict_pa_probabilities(
    pitcher,
    batter,
    p_throws,
    stand,
    inning=1,
    outs_when_up=0,
    balls=0,
    strikes=0,
    on_1b=0,
    on_2b=0,
    on_3b=0
):
    row = pd.DataFrame([{
        "pitcher": pitcher,
        "batter": batter,
        "p_throws": p_throws,
        "stand": stand,
        "inning": inning,
        "outs_when_up": outs_when_up,
        "balls": balls,
        "strikes": strikes,
        "on_1b": on_1b,
        "on_2b": on_2b,
        "on_3b": on_3b
    }])

    probs = model.predict_proba(row[features])[0]

    return dict(zip(classes, probs))

In [67]:
# ----------------------------
# Calibrate PA probabilities to more realistic MLB environment
# ----------------------------

LEAGUE_TARGETS = {
    "out": 0.58,
    "strikeout": 0.22,
    "walk": 0.08,
    "single": 0.12,
    "double": 0.045,
    "triple": 0.005,
    "hr": 0.03
}

# weights below 1.0 shrink a class
# weights above 1.0 boost a class
CALIBRATION_WEIGHTS = {
    "out": 1.12,
    "strikeout": 10.0,
    "walk": 18.0,
    "single": 0.34,
    "double": 0.50,
    "triple": 0.35,
    "hr": 0.80
}

def calibrate_probabilities(prob_dict, weights=CALIBRATION_WEIGHTS):
    adjusted = {}

    for k, v in prob_dict.items():
        adjusted[k] = float(v) * weights.get(k, 1.0)

    total = sum(adjusted.values())

    if total <= 0:
        return prob_dict

    adjusted = {k: v / total for k, v in adjusted.items()}
    return adjusted

In [68]:
# ----------------------------
# Example PA probability test
# ----------------------------

example = predict_pa_probabilities(
    pitcher=592789,
    batter=545361,
    p_throws="R",
    stand="L",
    inning=1,
    outs_when_up=0,
    balls=0,
    strikes=0,
    on_1b=0,
    on_2b=0,
    on_3b=0
)

example

{'double': np.float64(0.09909673846649344),
 'hr': np.float64(0.035437565001258324),
 'out': np.float64(0.5187773171807251),
 'single': np.float64(0.3295247431637175),
 'strikeout': np.float64(0.004526472284204226),
 'triple': np.float64(0.009796107400340289),
 'walk': np.float64(0.002841056503261094)}

In [69]:
# ----------------------------
# Sample one calibrated PA outcome
# ----------------------------

def sample_pa_outcome(prob_dict, calibrate=True):
    if calibrate:
        prob_dict = calibrate_probabilities(prob_dict)

    outcomes = list(prob_dict.keys())
    probs = np.array(list(prob_dict.values()), dtype=float)

    probs = probs / probs.sum()

    return np.random.choice(outcomes, p=probs)

In [70]:
# ----------------------------
# Compare raw vs calibrated probabilities
# ----------------------------

raw_probs = predict_pa_probabilities(
    pitcher=592789,
    batter=545361,
    p_throws="R",
    stand="L",
    inning=1,
    outs_when_up=0,
    balls=0,
    strikes=0,
    on_1b=0,
    on_2b=0,
    on_3b=0
)

cal_probs = calibrate_probabilities(raw_probs)

print("RAW")
for k, v in raw_probs.items():
    print(f"{k}: {round(float(v), 4)}")

print("\nCALIBRATED")
for k, v in cal_probs.items():
    print(f"{k}: {round(float(v), 4)}")

RAW
double: 0.0991
hr: 0.0354
out: 0.5188
single: 0.3295
strikeout: 0.0045
triple: 0.0098
walk: 0.0028

CALIBRATED
double: 0.0569
hr: 0.0326
out: 0.6672
single: 0.1287
strikeout: 0.052
triple: 0.0039
walk: 0.0587


In [71]:
# ----------------------------
# Stricter runner advancement logic
# ----------------------------

def apply_pa_outcome(outcome, bases, outs, runs):
    on_1b, on_2b, on_3b = bases

    # outs / strikeouts
    if outcome == "strikeout":
        outs += 1
        return (on_1b, on_2b, on_3b), outs, runs

    if outcome == "out":
        # add some double-play logic
        if outs < 2 and on_1b == 1:
            # chance of inning-killing DP
            dp_prob = 0.11 if on_2b == 0 else 0.08
            if np.random.rand() < dp_prob:
                outs += 2
                return (0, on_2b, on_3b), outs, runs

        outs += 1
        return (on_1b, on_2b, on_3b), outs, runs

    # walk
    if outcome == "walk":
        if on_1b and on_2b and on_3b:
            runs += 1
            return (1, 1, 1), outs, runs
        elif on_1b and on_2b:
            return (1, 1, 1), outs, runs
        elif on_1b:
            return (1, 1, on_3b), outs, runs
        else:
            return (1, on_2b, on_3b), outs, runs

    # single
    if outcome == "single":
        new_on_1b = 1
        new_on_2b = 0
        new_on_3b = 0

        # runner on 3rd scores
        if on_3b:
            runs += 1

        # runner on 2nd scores often, otherwise to 3rd
        if on_2b:
            if np.random.rand() < 0.58:
                runs += 1
            else:
                new_on_3b = 1

        # runner on 1st usually to 2nd, sometimes to 3rd
        if on_1b:
            if new_on_3b == 0 and np.random.rand() < 0.22:
                new_on_3b = 1
            else:
                new_on_2b = 1

        return (new_on_1b, new_on_2b, new_on_3b), outs, runs

    # double
    if outcome == "double":
        new_on_1b = 0
        new_on_2b = 1
        new_on_3b = 0

        # runners on 2nd and 3rd score
        runs += on_2b + on_3b

        # runner on 1st scores often, otherwise to 3rd
        if on_1b:
            if np.random.rand() < 0.62:
                runs += 1
            else:
                new_on_3b = 1

        return (new_on_1b, new_on_2b, new_on_3b), outs, runs

    # triple
    if outcome == "triple":
        runs += on_1b + on_2b + on_3b
        return (0, 0, 1), outs, runs

    # home run
    if outcome == "hr":
        runs += 1 + on_1b + on_2b + on_3b
        return (0, 0, 0), outs, runs

    # fallback
    outs += 1
    return (on_1b, on_2b, on_3b), outs, runs

In [72]:
# ----------------------------
# Simulate one half inning
# ----------------------------

def simulate_half_inning(lineup, pitcher_id, pitcher_hand, inning_num=1):
    outs = 0
    runs = 0
    bases = (0, 0, 0)
    lineup_index = 0

    while outs < 3:
        batter = lineup[lineup_index % len(lineup)]

        prob_dict = predict_pa_probabilities(
            pitcher=pitcher_id,
            batter=batter["batter_id"],
            p_throws=pitcher_hand,
            stand=batter["stand"],
            inning=inning_num,
            outs_when_up=outs,
            balls=0,
            strikes=0,
            on_1b=bases[0],
            on_2b=bases[1],
            on_3b=bases[2]
        )

        outcome = sample_pa_outcome(prob_dict)

        bases, outs, runs = apply_pa_outcome(outcome, bases, outs, runs)

        lineup_index += 1

    return runs, lineup_index

In [73]:
# ----------------------------
# Example lineup
# ----------------------------

home_lineup = [
    {"batter_id": 545361, "stand": "L"},
    {"batter_id": 608369, "stand": "R"},
    {"batter_id": 621043, "stand": "L"},
    {"batter_id": 592450, "stand": "R"},
    {"batter_id": 624413, "stand": "L"},
    {"batter_id": 543760, "stand": "R"},
    {"batter_id": 605141, "stand": "L"},
    {"batter_id": 668731, "stand": "R"},
    {"batter_id": 571448, "stand": "L"}
]

In [74]:
# ----------------------------
# Test one half inning
# ----------------------------

runs_scored, next_spot = simulate_half_inning(
    lineup=home_lineup,
    pitcher_id=592789,
    pitcher_hand="R",
    inning_num=1
)

print("Runs scored in half inning:", runs_scored)
print("Next lineup index:", next_spot)

Runs scored in half inning: 0
Next lineup index: 4


In [75]:
# ----------------------------
# Simulate one half inning
# Carry lineup position across innings
# ----------------------------

def simulate_half_inning(lineup, pitcher_id, pitcher_hand, start_index=0, inning_num=1):
    outs = 0
    runs = 0
    bases = (0, 0, 0)
    lineup_index = start_index

    while outs < 3:
        batter = lineup[lineup_index % len(lineup)]

        prob_dict = predict_pa_probabilities(
            pitcher=pitcher_id,
            batter=batter["batter_id"],
            p_throws=pitcher_hand,
            stand=batter["stand"],
            inning=inning_num,
            outs_when_up=outs,
            balls=0,
            strikes=0,
            on_1b=bases[0],
            on_2b=bases[1],
            on_3b=bases[2]
        )

        outcome = sample_pa_outcome(prob_dict)

        bases, outs, runs = apply_pa_outcome(outcome, bases, outs, runs)

        lineup_index += 1

    return runs, lineup_index

In [76]:
# ----------------------------
# Simulate one full game (no ties)
# ----------------------------

def simulate_game(
    away_lineup,
    home_lineup,
    away_pitcher_id,
    away_pitcher_hand,
    home_pitcher_id,
    home_pitcher_hand
):

    away_score = 0
    home_score = 0

    away_index = 0
    home_index = 0

    inning = 1

    while True:

        # AWAY HALF
        runs, away_index = simulate_half_inning(
            lineup=away_lineup,
            pitcher_id=home_pitcher_id,
            pitcher_hand=home_pitcher_hand,
            start_index=away_index,
            inning_num=inning
        )
        away_score += runs

        # HOME HALF
        runs, home_index = simulate_half_inning(
            lineup=home_lineup,
            pitcher_id=away_pitcher_id,
            pitcher_hand=away_pitcher_hand,
            start_index=home_index,
            inning_num=inning
        )
        home_score += runs

        # after 9 innings, check winner
        if inning >= 9:
            if home_score != away_score:
                break

        inning += 1

    return {
        "away_score": away_score,
        "home_score": home_score,
        "home_win": int(home_score > away_score),
        "away_win": int(away_score > home_score)
    }

In [77]:
# ----------------------------
# Test one full game
# ----------------------------

away_lineup = home_lineup.copy()

result = simulate_game(
    away_lineup=away_lineup,
    home_lineup=home_lineup,
    away_pitcher_id=592789,
    away_pitcher_hand="R",
    home_pitcher_id=605483,
    home_pitcher_hand="L"
)

result

{'away_score': 4, 'home_score': 2, 'home_win': 0, 'away_win': 1}

In [78]:
# ----------------------------
# Run many simulations
# ----------------------------

def run_monte_carlo(
    n_sims,
    away_lineup,
    home_lineup,
    away_pitcher_id,
    away_pitcher_hand,
    home_pitcher_id,
    home_pitcher_hand
):
    results = []

    for _ in range(n_sims):
        game_result = simulate_game(
            away_lineup=away_lineup,
            home_lineup=home_lineup,
            away_pitcher_id=away_pitcher_id,
            away_pitcher_hand=away_pitcher_hand,
            home_pitcher_id=home_pitcher_id,
            home_pitcher_hand=home_pitcher_hand
        )
        results.append(game_result)

    return pd.DataFrame(results)

In [81]:
# ----------------------------
# Load Statcast data for lineup building
# ----------------------------

statcast_path = "data/statcast_2021_2025.parquet"
sc = pd.read_parquet(statcast_path)

print(sc.shape)
sc.head()

(3846144, 119)


,pitch_type,game_date,release_speed,release_pos_x,release_pos_z,player_name,batter,pitcher,events,description,...,api_break_z_with_gravity,api_break_x_arm,api_break_x_batter_in,arm_angle,attack_angle,attack_direction,swing_path_tilt,intercept_ball_minus_batter_pos_x_inches,intercept_ball_minus_batter_pos_y_inches,season
0,FF,2021-11-02,93.7,1.39,6.72,"Smith, Will",493329,519293,field_out,hit_into_play,...,1.39,0.57,-0.57,46.9,<NA>,<NA>,<NA>,<NA>,<NA>,2021
1,FF,2021-11-02,92.9,1.38,6.72,"Smith, Will",493329,519293,None,foul,...,1.32,0.9,-0.9,48.0,<NA>,<NA>,<NA>,<NA>,<NA>,2021
2,FF,2021-11-02,93.1,1.35,6.73,"Smith, Will",493329,519293,None,called_strike,...,1.14,0.81,-0.81,46.7,<NA>,<NA>,<NA>,<NA>,<NA>,2021
3,FF,2021-11-02,94.6,1.31,6.73,"Smith, Will",670541,519293,field_out,hit_into_play,...,1.32,0.85,0.85,47.7,<NA>,<NA>,<NA>,<NA>,<NA>,2021
4,FF,2021-11-02,93.6,1.31,6.8,"Smith, Will",670541,519293,None,ball,...,1.2,0.9,0.9,48.9,<NA>,<NA>,<NA>,<NA>,<NA>,2021


In [82]:
# ----------------------------
# Check lineup-related columns
# ----------------------------

needed_cols = [
    "game_date",
    "game_pk",
    "inning_topbot",
    "batting_team",
    "fld_team",
    "batter",
    "stand",
    "pitcher",
    "p_throws",
    "at_bat_number"
]

existing_cols = [c for c in needed_cols if c in sc.columns]
missing_cols = [c for c in needed_cols if c not in sc.columns]

print("Existing:", existing_cols)
print("Missing:", missing_cols)

Existing: ['game_date', 'game_pk', 'inning_topbot', 'batter', 'stand', 'pitcher', 'p_throws', 'at_bat_number']
Missing: ['batting_team', 'fld_team']


In [85]:
# ----------------------------
# Build recent lineup candidates by team
# ----------------------------

lineup_df = sc.copy()

# derive batting_team if not already present
if "batting_team" not in lineup_df.columns:
    if all(col in lineup_df.columns for col in ["home_team", "away_team", "inning_topbot"]):
        lineup_df["batting_team"] = np.where(
            lineup_df["inning_topbot"] == "Top",
            lineup_df["away_team"],
            lineup_df["home_team"]
        )
    else:
        raise ValueError("Could not create batting_team. Need home_team, away_team, and inning_topbot columns.")

lineup_df = lineup_df.dropna(subset=["batting_team", "batter", "stand", "at_bat_number"])

lineup_df["game_date"] = pd.to_datetime(lineup_df["game_date"])

# keep only one row per batter per PA within a game
lineup_df = (
    lineup_df.sort_values(["game_date", "game_pk", "at_bat_number"])
             .drop_duplicates(subset=["game_pk", "batting_team", "batter", "at_bat_number"])
)

# estimate batting-order slot within each game
lineup_df["lineup_slot_est"] = (
    lineup_df.groupby(["game_pk", "batting_team"])["at_bat_number"]
             .rank(method="dense")
)

# cap to first 9 distinct lineup spots
lineup_df = lineup_df[lineup_df["lineup_slot_est"] <= 9].copy()

print(lineup_df.shape)
lineup_df.head()

(240078, 121)


,pitch_type,game_date,release_speed,release_pos_x,release_pos_z,player_name,batter,pitcher,events,description,...,api_break_x_batter_in,arm_angle,attack_angle,attack_direction,swing_path_tilt,intercept_ball_minus_batter_pos_x_inches,intercept_ball_minus_batter_pos_y_inches,season,batting_team,lineup_slot_est
765731,None,2021-03-15,<NA>,<NA>,<NA>,"Wilson, Bryse",650333,669060,single,hit_into_play,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2021,MIN,1.0
765730,None,2021-03-15,<NA>,<NA>,<NA>,"Wilson, Bryse",595909,669060,field_out,hit_into_play,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2021,MIN,2.0
765729,None,2021-03-15,<NA>,<NA>,<NA>,"Wilson, Bryse",680777,669060,field_out,hit_into_play,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2021,MIN,3.0
765728,None,2021-03-15,<NA>,<NA>,<NA>,"Wilson, Bryse",666135,669060,field_out,hit_into_play,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2021,MIN,4.0
765724,None,2021-03-15,<NA>,<NA>,<NA>,"Happ, J.A.",660670,457918,walk,ball,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2021,ATL,1.0


In [86]:
# ----------------------------
# Function: get likely lineup for a team
# ----------------------------

def get_team_lineup(team_code, as_of_date=None, lookback_days=30):
    df = lineup_df.copy()

    if as_of_date is not None:
        as_of_date = pd.to_datetime(as_of_date)
        start_date = as_of_date - pd.Timedelta(days=lookback_days)

        df = df[(df["game_date"] >= start_date) & (df["game_date"] <= as_of_date)].copy()

    df = df[df["batting_team"] == team_code].copy()

    if df.empty:
        raise ValueError(f"No lineup data found for team {team_code}")

    # summarize how often each batter appears and their average estimated slot
    summary = (
        df.groupby(["batter", "stand"], as_index=False)
          .agg(
              games_seen=("game_pk", "nunique"),
              mean_slot=("lineup_slot_est", "mean")
          )
    )

    # keep the most frequently seen players with best average slot stability
    summary = summary.sort_values(["mean_slot", "games_seen"], ascending=[True, False]).copy()

    # greedy fill of 9 lineup spots
    chosen = summary.head(9).copy()
    chosen = chosen.sort_values(["mean_slot", "games_seen"], ascending=[True, False]).reset_index(drop=True)

    lineup = []
    for _, row in chosen.iterrows():
        lineup.append({
            "batter_id": int(row["batter"]),
            "stand": row["stand"]
        })

    return lineup, chosen

In [87]:
# ----------------------------
# Example: build a real lineup
# ----------------------------

team_lineup, team_lineup_table = get_team_lineup(
    team_code="NYY",
    as_of_date="2025-09-30",
    lookback_days=30
)

print(team_lineup)
team_lineup_table

[{'batter_id': 663757, 'stand': 'L'}, {'batter_id': 592450, 'stand': 'R'}, {'batter_id': 641355, 'stand': 'L'}, {'batter_id': 596103, 'stand': 'R'}, {'batter_id': 700250, 'stand': 'L'}, {'batter_id': 502671, 'stand': 'R'}, {'batter_id': 519317, 'stand': 'R'}, {'batter_id': 642708, 'stand': 'R'}, {'batter_id': 665862, 'stand': 'L'}]


,batter,stand,games_seen,mean_slot
0,663757,L,22,1.636364
1,592450,R,27,2.333333
2,641355,L,27,3.333333
3,596103,R,4,3.5
4,700250,L,21,3.857143
5,502671,R,14,4.071429
6,519317,R,23,4.565217
7,642708,R,6,5.0
8,665862,L,23,5.869565


In [88]:
# ----------------------------
# Function: get likely starting pitcher for a team
# ----------------------------

def get_team_pitcher(team_code, as_of_date=None, lookback_days=30):
    df = sc.copy()

    # derive fielding team if not already present
    if "fld_team" not in df.columns:
        if all(col in df.columns for col in ["home_team", "away_team", "inning_topbot"]):
            df["fld_team"] = np.where(
                df["inning_topbot"] == "Top",
                df["home_team"],
                df["away_team"]
            )
        else:
            raise ValueError("Could not create fld_team. Need home_team, away_team, and inning_topbot columns.")

    df = df.dropna(subset=["fld_team", "pitcher", "p_throws", "game_date"])

    df["game_date"] = pd.to_datetime(df["game_date"])

    if as_of_date is not None:
        as_of_date = pd.to_datetime(as_of_date)
        start_date = as_of_date - pd.Timedelta(days=lookback_days)
        df = df[(df["game_date"] >= start_date) & (df["game_date"] <= as_of_date)].copy()

    df = df[df["fld_team"] == team_code].copy()

    if df.empty:
        raise ValueError(f"No pitcher data found for team {team_code}")

    pitcher_summary = (
        df.groupby(["pitcher", "p_throws"], as_index=False)
          .agg(
              pitches_seen=("pitcher", "size"),
              games_seen=("game_pk", "nunique")
          )
          .sort_values(["games_seen", "pitches_seen"], ascending=[False, False])
    )

    top_pitcher = pitcher_summary.iloc[0]

    return {
        "pitcher_id": int(top_pitcher["pitcher"]),
        "p_throws": top_pitcher["p_throws"]
    }, pitcher_summary

In [89]:
# ----------------------------
# Example: get likely pitcher
# ----------------------------

pitcher_info, pitcher_table = get_team_pitcher(
    team_code="BOS",
    as_of_date="2025-09-30",
    lookback_days=30
)

print(pitcher_info)
pitcher_table.head(10)

{'pitcher_id': 677161, 'p_throws': 'R'}


,pitcher,p_throws,pitches_seen,games_seen
12,677161,R,202,11
1,547973,L,192,11
14,686580,R,143,11
9,669711,R,162,10
2,571927,L,123,10
8,669684,L,226,9
10,676477,R,150,9
0,458677,L,145,9
11,676979,L,596,6
16,801139,L,220,6


In [90]:
# ----------------------------
# Example: simulate real team matchup
# ----------------------------

away_team = "BOS"
home_team = "NYY"
sim_date = "2025-09-30"

away_lineup, away_lineup_table = get_team_lineup(
    team_code=away_team,
    as_of_date=sim_date,
    lookback_days=30
)

home_lineup, home_lineup_table = get_team_lineup(
    team_code=home_team,
    as_of_date=sim_date,
    lookback_days=30
)

away_pitcher_info, _ = get_team_pitcher(
    team_code=away_team,
    as_of_date=sim_date,
    lookback_days=30
)

home_pitcher_info, _ = get_team_pitcher(
    team_code=home_team,
    as_of_date=sim_date,
    lookback_days=30
)

sim_results = run_monte_carlo(
    n_sims=1000,
    away_lineup=away_lineup,
    home_lineup=home_lineup,
    away_pitcher_id=away_pitcher_info["pitcher_id"],
    away_pitcher_hand=away_pitcher_info["p_throws"],
    home_pitcher_id=home_pitcher_info["pitcher_id"],
    home_pitcher_hand=home_pitcher_info["p_throws"]
)

home_win_prob = sim_results["home_win"].mean()
away_win_prob = sim_results["away_win"].mean()
avg_home_score = sim_results["home_score"].mean()
avg_away_score = sim_results["away_score"].mean()

print("Away team:", away_team)
print("Home team:", home_team)
print("Home win probability:", round(home_win_prob, 4))
print("Away win probability:", round(away_win_prob, 4))
print("Average home score:", round(avg_home_score, 2))
print("Average away score:", round(avg_away_score, 2))

Away team: BOS
Home team: NYY
Home win probability: 0.441
Away win probability: 0.559
Average home score: 2.26
Average away score: 2.66


In [ ]:
# ----------------------------
# Single game simulation test
# ----------------------------

sim_results = run_monte_carlo(
    n_sims=1000,
    away_lineup=away_lineup,
    home_lineup=home_lineup,
    away_pitcher_id=592789,
    away_pitcher_hand="R",
    home_pitcher_id=605483,
    home_pitcher_hand="L"
)

sim_results.head()

,away_score,home_score,home_win,away_win
0,3,0,0,1
1,0,2,1,0
2,2,3,1,0
3,0,1,1,0
4,10,0,0,1


In [ ]:
# ----------------------------
# Single game summary
# ----------------------------

home_win_prob = sim_results["home_win"].mean()
away_win_prob = sim_results["away_win"].mean()

avg_home_score = sim_results["home_score"].mean()
avg_away_score = sim_results["away_score"].mean()

print("Home win probability:", round(home_win_prob, 4))
print("Away win probability:", round(away_win_prob, 4))
print("Average home score:", round(avg_home_score, 2))
print("Average away score:", round(avg_away_score, 2))

Home win probability: 0.467
Away win probability: 0.533
Average home score: 2.38
Average away score: 2.56


In [91]:
# ----------------------------
# Build schedule for a chosen date
# ----------------------------

def get_schedule_for_date(sim_date):
    df = sc.copy()

    df["game_date"] = pd.to_datetime(df["game_date"])

    sim_date = pd.to_datetime(sim_date)

    needed = ["game_date", "game_pk", "home_team", "away_team"]
    missing = [c for c in needed if c not in df.columns]

    if missing:
        raise ValueError(f"Missing required columns for schedule build: {missing}")

    sched = (
        df[df["game_date"] == sim_date][needed]
        .drop_duplicates(subset=["game_pk", "home_team", "away_team"])
        .sort_values(["game_pk"])
        .reset_index(drop=True)
    )

    return sched

In [92]:
# ----------------------------
# Example schedule lookup
# ----------------------------

schedule_df = get_schedule_for_date("2025-09-30")
schedule_df

,game_date,game_pk,home_team,away_team
0,2025-09-30,813066,CHC,SD
1,2025-09-30,813069,LAD,CIN
2,2025-09-30,813072,CLE,DET
3,2025-09-30,813074,NYY,BOS


In [93]:
# ----------------------------
# Simulate one scheduled game
# ----------------------------

def simulate_scheduled_game(row, n_sims=500, lookback_days=30):
    away_team = row["away_team"]
    home_team = row["home_team"]
    sim_date = pd.to_datetime(row["game_date"])

    away_lineup, _ = get_team_lineup(
        team_code=away_team,
        as_of_date=sim_date,
        lookback_days=lookback_days
    )

    home_lineup, _ = get_team_lineup(
        team_code=home_team,
        as_of_date=sim_date,
        lookback_days=lookback_days
    )

    away_pitcher_info, _ = get_team_pitcher(
        team_code=away_team,
        as_of_date=sim_date,
        lookback_days=lookback_days
    )

    home_pitcher_info, _ = get_team_pitcher(
        team_code=home_team,
        as_of_date=sim_date,
        lookback_days=lookback_days
    )

    sim_results = run_monte_carlo(
        n_sims=n_sims,
        away_lineup=away_lineup,
        home_lineup=home_lineup,
        away_pitcher_id=away_pitcher_info["pitcher_id"],
        away_pitcher_hand=away_pitcher_info["p_throws"],
        home_pitcher_id=home_pitcher_info["pitcher_id"],
        home_pitcher_hand=home_pitcher_info["p_throws"]
    )

    home_win_prob = sim_results["home_win"].mean()
    away_win_prob = sim_results["away_win"].mean()
    avg_home_score = sim_results["home_score"].mean()
    avg_away_score = sim_results["away_score"].mean()

    return {
        "game_date": sim_date.date(),
        "away_team": away_team,
        "home_team": home_team,
        "away_pitcher_id": away_pitcher_info["pitcher_id"],
        "home_pitcher_id": home_pitcher_info["pitcher_id"],
        "away_win_prob": round(float(away_win_prob), 4),
        "home_win_prob": round(float(home_win_prob), 4),
        "avg_away_score": round(float(avg_away_score), 2),
        "avg_home_score": round(float(avg_home_score), 2),
        "favorite": away_team if away_win_prob > home_win_prob else home_team
    }

In [94]:
# ----------------------------
# Test one game from the schedule
# ----------------------------

one_game_result = simulate_scheduled_game(schedule_df.iloc[0], n_sims=250)
one_game_result

{'game_date': datetime.date(2025, 9, 30),
 'away_team': 'SD',
 'home_team': 'CHC',
 'away_pitcher_id': 669093,
 'home_pitcher_id': 519141,
 'away_win_prob': 0.58,
 'home_win_prob': 0.42,
 'avg_away_score': 2.67,
 'avg_home_score': 2.13,
 'favorite': 'SD'}

In [95]:
# ----------------------------
# Simulate full slate for a date
# ----------------------------

def simulate_full_slate(sim_date, n_sims=250, lookback_days=30):
    sched = get_schedule_for_date(sim_date)

    results = []

    for _, row in sched.iterrows():
        try:
            result = simulate_scheduled_game(
                row=row,
                n_sims=n_sims,
                lookback_days=lookback_days
            )
            results.append(result)
        except Exception as e:
            results.append({
                "game_date": pd.to_datetime(row["game_date"]).date(),
                "away_team": row["away_team"],
                "home_team": row["home_team"],
                "error": str(e)
            })

    return pd.DataFrame(results)

In [96]:
# ----------------------------
# Example full slate simulation
# ----------------------------

slate_results = simulate_full_slate(
    sim_date="2025-09-30",
    n_sims=250,
    lookback_days=30
)

slate_results

,game_date,away_team,home_team,away_pitcher_id,home_pitcher_id,away_win_prob,home_win_prob,avg_away_score,avg_home_score,favorite
0,2025-09-30,SD,CHC,669093,519141,0.676,0.324,3.00,1.92,SD
1,2025-09-30,CIN,LAD,641941,595014,0.516,0.484,2.14,2.00,CIN
2,2025-09-30,DET,CLE,592454,671922,0.536,0.464,2.24,2.16,DET
3,2025-09-30,BOS,NYY,677161,518585,0.564,0.436,2.91,2.36,BOS


In [97]:
# ----------------------------
# Sort slate by biggest edge
# ----------------------------

clean_slate = slate_results.copy()

if "error" in clean_slate.columns:
    clean_slate = clean_slate[clean_slate["error"].isna()].copy()

clean_slate["fav_win_prob"] = clean_slate[["away_win_prob", "home_win_prob"]].max(axis=1)
clean_slate = clean_slate.sort_values("fav_win_prob", ascending=False)

clean_slate[
    ["game_date", "away_team", "home_team", "away_win_prob", "home_win_prob", "avg_away_score", "avg_home_score", "favorite", "fav_win_prob"]
]

,game_date,away_team,home_team,away_win_prob,home_win_prob,avg_away_score,avg_home_score,favorite,fav_win_prob
0,2025-09-30,SD,CHC,0.676,0.324,3.00,1.92,SD,0.676
3,2025-09-30,BOS,NYY,0.564,0.436,2.91,2.36,BOS,0.564
2,2025-09-30,DET,CLE,0.536,0.464,2.24,2.16,DET,0.536
1,2025-09-30,CIN,LAD,0.516,0.484,2.14,2.00,CIN,0.516
